# Part 1 - A French lemmatizer from scratch

**Ran Lu**

This notebook implements and evaluates a French lemmatizer without calling an existing
lemmatizer inside the proposed system. Two input settings are studied:

1. **with POS**: the token and its gold Universal POS tag are available;
2. **without POS**: only the surface token is available.

The final comparison uses spaCy's `fr_core_news_sm` as an external baseline. All reported
test results are computed on a held-out split that is not used for either rule learning or
hyperparameter selection.

## 1. Choice of approach

I chose a **dictionary plus learned suffix rules** approach rather than a neural model.
French lemmatization contains many frequent irregular mappings (`suis` -> `etre`) that a
dictionary handles well, while productive inflections (`-aient`, `-es`, `-aux`, etc.) need
a fallback for unseen forms. The hybrid system therefore has three levels:

1. return the most frequent lemma of an observed surface form;
2. retry with a case-folded dictionary;
3. for an unseen form, apply the longest reliable suffix-replacement rule learned from the
   training set, otherwise return the case-folded token unchanged.

In the POS-aware version all dictionary entries and rules are conditioned on the Universal
POS tag. In the POS-free version they are not. Capitalisation and digits are still observable
surface features, so unseen capitalised or numeric tokens are conservatively preserved.

This is a corpus-derived rule system: no lemma list, conjugation library, spaCy component,
or test-set annotation is consulted during fitting. Its main advantages are interpretability,
fast inference, and a meaningful ablation of the POS information.

## 2. Data and evaluation protocol

I use **Universal Dependencies French GSD v2.16**, whose train/dev/test split contains
real French sentences with gold forms, lemmas and Universal POS tags. Multi-word-token
range lines and empty nodes are skipped, but their syntactic words are retained. Accuracy
is exact lemma match; punctuation and symbols are excluded because copying them would make
the score artificially easier.

The dev set selects the minimum rule support and precision. The test set is opened only
after those values are fixed. I also report accuracy on true OOV surface forms (case-folded
forms absent from training), where suffix generalisation actually matters.

Data: [UD French GSD](https://github.com/UniversalDependencies/UD_French-GSD), release
`r2.16`. Baseline: spaCy `fr_core_news_sm` 3.8.0.

In [1]:
from collections import Counter, defaultdict
from pathlib import Path
from urllib.request import urlretrieve
import platform

import pandas as pd

DATA_DIR = Path("data/ud_french_gsd")
DATA_DIR.mkdir(parents=True, exist_ok=True)
BASE_URL = "https://raw.githubusercontent.com/UniversalDependencies/UD_French-GSD/r2.16"
FILES = {
    split: f"fr_gsd-ud-{split}.conllu"
    for split in ("train", "dev", "test")
}

for filename in FILES.values():
    path = DATA_DIR / filename
    if not path.exists():
        urlretrieve(f"{BASE_URL}/{filename}", path)

print("Python:", platform.python_version())
print("Data directory:", DATA_DIR.resolve())

Python: 3.9.20
Data directory: /Users/luran/Documents/ChatGPT/NLP Today/data/ud_french_gsd


In [2]:
def read_conllu(path):
    sentences, sentence = [], []
    with open(path, encoding="utf-8") as stream:
        for line in stream:
            line = line.rstrip("\n")
            if not line:
                if sentence:
                    sentences.append(sentence)
                    sentence = []
                continue
            if line.startswith("#"):
                continue
            fields = line.split("\t")
            token_id = fields[0]
            if "-" in token_id or "." in token_id:
                continue
            form, lemma, upos = fields[1], fields[2], fields[3]
            if lemma != "_":
                sentence.append((form, lemma, upos))
    if sentence:
        sentences.append(sentence)
    return sentences


datasets = {
    split: read_conllu(DATA_DIR / filename)
    for split, filename in FILES.items()
}

data_summary = pd.DataFrame([
    {
        "split": split,
        "sentences": len(sentences),
        "tokens": sum(len(sentence) for sentence in sentences),
        "unique forms": len({form.casefold() for sentence in sentences for form, _, _ in sentence}),
    }
    for split, sentences in datasets.items()
])
data_summary

,split,sentences,tokens,unique forms
0,train,14450,354617,39679
1,dev,1476,35714,8846
2,test,416,10015,3121


## 3. Implementation

A suffix rule is represented as `(old_suffix -> new_suffix)`. For example, if the training
pair shares the stem `chant` in `chantaient -> chanter`, it proposes `aient -> er`. The split
is found by the longest common prefix, so rules can both delete and insert characters.

A rule is retained only if its most common output has enough training support and empirical
precision. At prediction time rules are tried from longest to shortest. Conditioning keys on
POS prevents, for example, a nominal ending from competing directly with a verbal ending.
`PROPN` and `NUM` are preserved only after both exact and case-folded dictionary lookup fail.

In [3]:
def most_frequent(counter):
    return max(counter.items(), key=lambda item: (item[1], item[0]))[0]


def common_prefix_length(left, right):
    length = 0
    while length < min(len(left), len(right)) and left[length] == right[length]:
        length += 1
    return length


class FrenchRuleLemmatizer:
    def __init__(self, use_pos=True, min_support=5, min_precision=0.70, max_suffix=8):
        self.use_pos = use_pos
        self.min_support = min_support
        self.min_precision = min_precision
        self.max_suffix = max_suffix

    def fit(self, sentences):
        exact_counts = defaultdict(Counter)
        lower_counts = defaultdict(Counter)
        rule_counts = defaultdict(Counter)

        for sentence in sentences:
            for form, lemma, pos in sentence:
                exact_key = (form, pos) if self.use_pos else form
                lower_form, lower_lemma = form.casefold(), lemma.casefold()
                lower_key = (lower_form, pos) if self.use_pos else lower_form
                exact_counts[exact_key][lemma] += 1
                lower_counts[lower_key][lower_lemma] += 1

                if pos in {"PUNCT", "SYM"}:
                    continue
                prefix = common_prefix_length(lower_form, lower_lemma)
                old_suffix = lower_form[prefix:]
                new_suffix = lower_lemma[prefix:]
                if len(old_suffix) <= self.max_suffix:
                    rule_key = (pos, old_suffix) if self.use_pos else old_suffix
                    rule_counts[rule_key][new_suffix] += 1

        self.exact_ = {key: most_frequent(counts) for key, counts in exact_counts.items()}
        self.lower_ = {key: most_frequent(counts) for key, counts in lower_counts.items()}
        self.rules_ = {}
        self.rule_stats_ = {}
        for key, counts in rule_counts.items():
            replacement = most_frequent(counts)
            support = counts[replacement]
            precision = support / sum(counts.values())
            if support >= self.min_support and precision >= self.min_precision:
                self.rules_[key] = replacement
                self.rule_stats_[key] = (support, precision)
        return self

    def predict_one(self, form, pos=None):
        if self.use_pos and pos is None:
            raise ValueError("A POS tag is required when use_pos=True")

        exact_key = (form, pos) if self.use_pos else form
        if exact_key in self.exact_:
            return self.exact_[exact_key]

        lower_form = form.casefold()
        lower_key = (lower_form, pos) if self.use_pos else lower_form
        if lower_key in self.lower_:
            return self.lower_[lower_key]

        if (self.use_pos and pos in {"PROPN", "NUM"}) or (
            not self.use_pos and (form[:1].isupper() or any(char.isdigit() for char in form))
        ):
            return form

        for size in range(min(self.max_suffix, len(lower_form)), -1, -1):
            suffix = lower_form[-size:] if size else ""
            rule_key = (pos, suffix) if self.use_pos else suffix
            if rule_key in self.rules_:
                stem = lower_form[:-size] if size else lower_form
                return stem + self.rules_[rule_key]
        return lower_form

    def predict(self, forms, pos_tags=None):
        if pos_tags is None:
            pos_tags = [None] * len(forms)
        return [self.predict_one(form, pos) for form, pos in zip(forms, pos_tags)]

In [4]:
EXCLUDED_POS = {"PUNCT", "SYM"}


def evaluation_rows(sentences):
    return [
        (form, lemma, pos)
        for sentence in sentences
        for form, lemma, pos in sentence
        if pos not in EXCLUDED_POS
    ]


def model_accuracy(model, rows):
    correct = sum(
        model.predict_one(form, pos if model.use_pos else None) == lemma
        for form, lemma, pos in rows
    )
    return correct / len(rows)


train_forms = {
    form.casefold()
    for sentence in datasets["train"]
    for form, _, _ in sentence
}
dev_rows = evaluation_rows(datasets["dev"])
test_rows = evaluation_rows(datasets["test"])

## 4. Development-set selection

`max_suffix=8` covers the common French inflectional endings while avoiding extremely long,
word-specific pseudo-rules. I compare a small, declared grid for support and precision. The
selected configuration maximises the mean of the with-POS and without-POS dev accuracies;
no test result participates in this choice.

In [5]:
candidates = [(2, 0.60), (3, 0.60), (3, 0.70), (5, 0.60), (5, 0.70)]
tuning_records = []

for support, precision in candidates:
    row = {"min_support": support, "min_precision": precision}
    for use_pos, label in [(True, "with POS"), (False, "without POS")]:
        candidate = FrenchRuleLemmatizer(
            use_pos=use_pos,
            min_support=support,
            min_precision=precision,
            max_suffix=8,
        ).fit(datasets["train"])
        row[label] = model_accuracy(candidate, dev_rows)
        row[f"rules ({label})"] = len(candidate.rules_)
    row["mean accuracy"] = (row["with POS"] + row["without POS"]) / 2
    tuning_records.append(row)

tuning = pd.DataFrame(tuning_records).sort_values("mean accuracy", ascending=False)
tuning.style.format({
    "min_precision": "{:.2f}",
    "with POS": "{:.3%}",
    "without POS": "{:.3%}",
    "mean accuracy": "{:.3%}",
})

,min_support,min_precision,with POS,rules (with POS),without POS,rules (without POS),mean accuracy
4,5,0.70,97.925%,264,95.645%,202,96.785%
3,5,0.60,97.931%,276,95.516%,208,96.724%
2,3,0.70,97.303%,333,95.630%,245,96.466%
1,3,0.60,97.306%,345,95.312%,254,96.309%
0,2,0.60,97.287%,413,95.253%,307,96.270%


In [6]:
selected = tuning.iloc[0]
SELECTED_SUPPORT = int(selected["min_support"])
SELECTED_PRECISION = float(selected["min_precision"])

proposed_with_pos = FrenchRuleLemmatizer(
    use_pos=True,
    min_support=SELECTED_SUPPORT,
    min_precision=SELECTED_PRECISION,
).fit(datasets["train"])
proposed_without_pos = FrenchRuleLemmatizer(
    use_pos=False,
    min_support=SELECTED_SUPPORT,
    min_precision=SELECTED_PRECISION,
).fit(datasets["train"])

print(f"Selected min_support={SELECTED_SUPPORT}, min_precision={SELECTED_PRECISION:.2f}")
print(f"POS-aware rules: {len(proposed_with_pos.rules_):,}")
print(f"POS-free rules:  {len(proposed_without_pos.rules_):,}")

Selected min_support=5, min_precision=0.70
POS-aware rules: 264
POS-free rules:  202


The most frequent learned rules below provide a sanity check. The empty suffix is a default
identity-like transformation; longer rules take priority at inference.

In [7]:
rule_examples = []
for (pos, suffix), replacement in proposed_with_pos.rules_.items():
    support, precision = proposed_with_pos.rule_stats_[(pos, suffix)]
    rule_examples.append({
        "POS": pos,
        "old suffix": suffix or "<empty>",
        "new suffix": replacement or "<empty>",
        "support": support,
        "precision": precision,
    })

pd.DataFrame(rule_examples).sort_values(
    ["support", "precision"], ascending=False
).head(15).style.format({"precision": "{:.1%}"})

,POS,old suffix,new suffix,support,precision
5,ADP,,,51655,100.0%
7,NOUN,,,50987,99.8%
15,PROPN,,,24674,100.0%
6,DET,,,19921,99.9%
1,NOUN,s,,14670,100.0%
4,ADV,,,11585,100.0%
16,ADJ,,,10615,99.9%
11,DET,a,e,9710,92.1%
22,CCONJ,,,9295,100.0%
49,NUM,,,9232,100.0%


## 5. spaCy baselines

I use fixed UD token boundaries for every system, so this experiment measures lemmatisation
rather than tokenisation. Two spaCy conditions mirror the two inputs:

- **with POS**: assign the gold Universal POS tag and call only spaCy's French rule
  lemmatiser;
- **without POS**: provide only words and run spaCy's normal `tok2vec`, `morphologizer`,
  `attribute_ruler`, and `lemmatizer` components. POS is therefore predicted internally.

The second condition gives spaCy more modelling capacity than the proposed POS-free rules,
but both start from the same observable input.

In [8]:
try:
    import spacy
    from spacy.tokens import Doc
    nlp = spacy.load("fr_core_news_sm")
except (ImportError, OSError) as error:
    raise RuntimeError(
        "Install the baseline with: pip install spacy==3.8.2 "
        "https://github.com/explosion/spacy-models/releases/download/"
        "fr_core_news_sm-3.8.0/fr_core_news_sm-3.8.0-py3-none-any.whl"
    ) from error

print("spaCy:", spacy.__version__)
print("Pipeline:", nlp.pipe_names)

spaCy: 3.8.2
Pipeline: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [9]:
def spacy_gold_pos_predictions(sentences, nlp):
    predictions = []
    lemmatizer = nlp.get_pipe("lemmatizer")
    for sentence in sentences:
        doc = Doc(nlp.vocab, words=[form for form, _, _ in sentence])
        for token, (_, _, pos) in zip(doc, sentence):
            token.pos = nlp.vocab.strings.add(pos)
        lemmatizer(doc)
        predictions.extend(token.lemma_ for token in doc)
    return predictions


def spacy_predicted_pos_outputs(sentences, nlp):
    docs = (
        Doc(nlp.vocab, words=[form for form, _, _ in sentence])
        for sentence in sentences
    )
    lemmas, pos_tags = [], []
    for doc in nlp.pipe(docs, batch_size=64, disable=["parser", "ner"]):
        lemmas.extend(token.lemma_ for token in doc)
        pos_tags.extend(token.pos_ for token in doc)
    return lemmas, pos_tags


raw_test_rows = [item for sentence in datasets["test"] for item in sentence]
spacy_gold_raw = spacy_gold_pos_predictions(datasets["test"], nlp)
spacy_pred_raw, spacy_pos_raw = spacy_predicted_pos_outputs(datasets["test"], nlp)

keep_indices = [
    index for index, (_, _, pos) in enumerate(raw_test_rows)
    if pos not in EXCLUDED_POS
]
spacy_gold = [spacy_gold_raw[index] for index in keep_indices]
spacy_pred = [spacy_pred_raw[index] for index in keep_indices]
spacy_predicted_pos = [spacy_pos_raw[index] for index in keep_indices]

## 6. Held-out test results

The primary metric is token-level exact-match accuracy. OOV accuracy uses only tokens whose
case-folded surface form never occurs in training. `Delta vs spaCy` compares systems under
the same input condition.

In [10]:
forms = [form for form, _, _ in test_rows]
gold_lemmas = [lemma for _, lemma, _ in test_rows]
gold_pos = [pos for _, _, pos in test_rows]

proposed_pos_pred = proposed_with_pos.predict(forms, gold_pos)
proposed_no_pos_pred = proposed_without_pos.predict(forms)
oov_mask = [form.casefold() not in train_forms for form in forms]


def exact_accuracy(gold, predicted, mask=None):
    if mask is None:
        mask = [True] * len(gold)
    selected = [(g, p) for g, p, use in zip(gold, predicted, mask) if use]
    return sum(g == p for g, p in selected) / len(selected)


result_records = [
    {"system": "Proposed", "input": "gold POS", "pred": proposed_pos_pred},
    {"system": "spaCy", "input": "gold POS", "pred": spacy_gold},
    {"system": "Proposed", "input": "no POS", "pred": proposed_no_pos_pred},
    {"system": "spaCy", "input": "no POS (predicted internally)", "pred": spacy_pred},
]
results = pd.DataFrame([
    {
        "system": record["system"],
        "input": record["input"],
        "test tokens": len(gold_lemmas),
        "accuracy": exact_accuracy(gold_lemmas, record["pred"]),
        "OOV tokens": sum(oov_mask),
        "OOV accuracy": exact_accuracy(gold_lemmas, record["pred"], oov_mask),
    }
    for record in result_records
])
results.style.format({"accuracy": "{:.3%}", "OOV accuracy": "{:.3%}"})

,system,input,test tokens,accuracy,OOV tokens,OOV accuracy
0,Proposed,gold POS,8790,97.964%,541,73.198%
1,spaCy,gold POS,8790,94.346%,541,92.606%
2,Proposed,no POS,8790,95.586%,541,59.889%
3,spaCy,no POS (predicted internally),8790,90.922%,541,76.895%


In [11]:
pos_prediction_accuracy = exact_accuracy(gold_pos, spacy_predicted_pos)
pos_gain = exact_accuracy(gold_lemmas, proposed_pos_pred) - exact_accuracy(
    gold_lemmas, proposed_no_pos_pred
)
print(f"spaCy internal POS accuracy: {pos_prediction_accuracy:.3%}")
print(f"Proposed-system gain from gold POS: {pos_gain:+.3%} points")

spaCy internal POS accuracy: 91.889%
Proposed-system gain from gold POS: +2.378% points


### Breakdown by POS

This table separates the effect of POS from corpus frequency. Rare classes have less stable
percentages, so the token count is shown beside every score.

In [12]:
per_pos_records = []
for pos in sorted(set(gold_pos)):
    mask = [tag == pos for tag in gold_pos]
    per_pos_records.append({
        "POS": pos,
        "tokens": sum(mask),
        "Proposed +POS": exact_accuracy(gold_lemmas, proposed_pos_pred, mask),
        "Proposed -POS": exact_accuracy(gold_lemmas, proposed_no_pos_pred, mask),
        "spaCy +POS": exact_accuracy(gold_lemmas, spacy_gold, mask),
        "spaCy -POS": exact_accuracy(gold_lemmas, spacy_pred, mask),
    })

per_pos = pd.DataFrame(per_pos_records).sort_values("tokens", ascending=False)
per_pos.style.format({column: "{:.1%}" for column in per_pos.columns if column not in {"POS", "tokens"}})

,POS,tokens,Proposed +POS,Proposed -POS,spaCy +POS,spaCy -POS
7,NOUN,1870,97.3%,92.9%,96.7%,93.9%
1,ADP,1479,100.0%,100.0%,98.4%,98.3%
5,DET,1479,100.0%,98.9%,96.1%,95.5%
12,VERB,821,93.8%,89.3%,96.0%,90.5%
0,ADJ,609,91.8%,86.2%,94.3%,87.8%
9,PRON,559,99.6%,97.0%,53.5%,53.3%
2,ADV,487,99.0%,98.4%,98.8%,97.1%
10,PROPN,487,98.8%,93.6%,99.2%,70.4%
3,AUX,359,99.7%,99.7%,100.0%,99.7%
4,CCONJ,249,100.0%,99.6%,99.2%,98.8%


### Error inspection

The sample is deterministic and prioritises OOV errors. It reveals whether the aggregate
score is hiding genuine morphological failures, annotation conventions, or proper-name
casing errors.

In [13]:
error_rows = []
for form, lemma, pos, prediction, is_oov in zip(
    forms, gold_lemmas, gold_pos, proposed_pos_pred, oov_mask
):
    if prediction != lemma:
        error_rows.append({
            "form": form,
            "gold lemma": lemma,
            "prediction": prediction,
            "POS": pos,
            "OOV": is_oov,
        })

pd.DataFrame(error_rows).sort_values(
    ["OOV", "POS", "form"], ascending=[False, True, True]
).head(20)

,form,gold lemma,prediction,POS,OOV
47,150e,150e,150,ADJ,True
62,74e,74e,74,ADJ,True
141,8ème,8ème,8èm,ADJ,True
64,Canéjanaise,canéjanais,canéjanaix,ADJ,True
57,Rochelais,rochelais,rochelai,ADJ,True
133,adaptable,adaptable,adaptab,ADJ,True
11,anti-hérétiques,anti-hérétique,anti-hérétic,ADJ,True
121,antipathiques,antipathique,antipathic,ADJ,True
167,arabophone,arabophone,arabopho,ADJ,True
20,avantgardiste,avant-gardiste,avantgardis,ADJ,True


## 7. Conclusion

The experiment supports the dictionary-plus-rules choice for this in-domain setting. The
training dictionary solves frequent irregular forms, while learned suffix replacements give
non-trivial OOV coverage. Supplying gold POS improves the proposed system, especially where
the same spelling can be nominal, adjectival, or verbal. The POS-free version remains useful
because its interface needs only a token and its orthographic features.

On the 8,790 evaluated GSD test tokens, the proposed system reaches **97.964% with gold POS**
and **95.586% without POS**, versus spaCy's **94.346%** and **90.922%**, respectively. This
should not be interpreted as a universal superiority claim: the proposed dictionary is fitted
directly to GSD's training domain and annotation conventions, whereas spaCy is a
general-purpose pipeline. On the 541 genuinely unseen surface forms, the direction reverses:
the proposed system obtains **73.198% with POS** and **59.889% without POS**, while spaCy
obtains **92.606%** and **76.895%**. The overall lead is therefore driven mainly by strong
in-domain lexical coverage; spaCy generalises better to unseen forms.

The remaining errors are concentrated in genuinely unseen irregular forms, ambiguous short
forms, derivations for which a suffix edit is insufficient, and corpus-specific lemma choices
(notably some pronouns and proper names). A stronger next model would predict character edit
scripts with a sequence model and combine them with the same high-precision dictionary; an
out-of-domain corpus would then be necessary to test robustness.

## References

- Guillaume et al. Universal Dependencies French GSD, release 2.16.
- Honnibal et al. (2020). *spaCy: Industrial-strength Natural Language Processing in Python*.